# 02 Feature Engineering
**AR Risk Scoring Project | Finance Analytics**

Goal: build and validate all engineered features used in the model.
Features are defined in `sql/03_feature_engineering.sql` (SQL layer)
and extended here in Python (pandas layer).


In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install",
                "pandas", "numpy", "matplotlib", "seaborn", "-q"])


In [ ]:
import sqlite3, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams["figure.dpi"] = 120

sys.path.append("../src")
from etl import run as run_etl

DB_PATH = Path("../data/processed/ar_risk.db")
if not DB_PATH.exists():
    run_etl()

with sqlite3.connect(DB_PATH) as conn:
    df = pd.read_sql("SELECT * FROM v_model_features", conn)

print(f"Features loaded: {df.shape}")
df.head()


## 1. SQL-layer features (from `v_model_features` view)

In [ ]:
# These were created in sql/03_feature_engineering.sql
sql_features = ["weighted_late_score", "utilisation_ratio", "income_band", "dependents_income_ratio"]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, col in zip(axes, sql_features):
    data = df[col].dropna().clip(upper=df[col].quantile(0.99))
    ax.hist(data, bins=35, color="#4C9BE8", edgecolor="white", alpha=0.85)
    ax.set_title(col.replace("_"," ").title(), fontsize=10)
    ax.set_xlabel("Value")

plt.suptitle("SQL-engineered Features", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


## 2. Python-layer features

In [ ]:
# ── Weighted late score breakdown ────────────────────────────────────────────
# Already in SQL — validate correlation with target
fig, ax = plt.subplots(figsize=(8, 4))
on_time = df[df["target"]==0]["weighted_late_score"].clip(upper=15)
default = df[df["target"]==1]["weighted_late_score"].clip(upper=15)
ax.hist(on_time, bins=20, alpha=0.6, color="#4C9BE8", label="On Time", density=True)
ax.hist(default, bins=20, alpha=0.6, color="#E8674C", label="Default", density=True)
ax.set_title("Weighted Late Score by Class", fontsize=12, fontweight="bold")
ax.set_xlabel("Weighted Late Score")
ax.set_ylabel("Density")
ax.legend()
plt.tight_layout()
plt.show()

sep = df.groupby("target")["weighted_late_score"].median()
print(f"Median weighted_late_score — On Time: {sep[0]:.2f} | Default: {sep[1]:.2f}")


In [ ]:
# ── Python additional features ───────────────────────────────────────────────
def add_python_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # 1. Any late event flag (binary)
    df["has_any_late"] = (df["weighted_late_score"] > 0).astype(int)

    # 2. Log-transformed income (reduces skew)
    df["log_income"] = np.log1p(df["monthly_income"].fillna(df["monthly_income"].median()))

    # 3. Age risk flag: under 30 or over 65
    df["age_risk_flag"] = ((df["age"] < 30) | (df["age"] > 65)).astype(int)

    # 4. Credit lines per real estate loan (leverage ratio)
    df["credit_lines_per_loan"] = df["num_open_credit_lines"] / (df["num_real_estate_loans"] + 1)

    # 5. High debt + late payment (interaction feature)
    median_debt = df["debt_ratio"].median()
    df["high_debt_and_late"] = (
        (df["debt_ratio"] > median_debt) & (df["has_any_late"] == 1)
    ).astype(int)

    return df

df_eng = add_python_features(df)
new_cols = ["has_any_late","log_income","age_risk_flag","credit_lines_per_loan","high_debt_and_late"]
print("New features added:")
print(df_eng[new_cols].describe().T[["mean","std","min","max"]].round(4).to_string())


## 3. Feature importance preview (mutual information)

In [ ]:
from sklearn.feature_selection import mutual_info_classif

all_features = [
    "credit_limit","age","debt_ratio","monthly_income",
    "num_open_credit_lines","num_real_estate_loans","num_dependents",
    "weighted_late_score","utilisation_ratio","income_band","dependents_income_ratio",
    "has_any_late","log_income","age_risk_flag","credit_lines_per_loan","high_debt_and_late",
]

X = df_eng[all_features].fillna(0)
y = df_eng["target"]

mi = mutual_info_classif(X, y, random_state=42)
mi_series = pd.Series(mi, index=all_features).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
bars = ax.barh(mi_series.index, mi_series.values, color="#4C9BE8", edgecolor="white")
ax.set_xlabel("Mutual Information Score")
ax.set_title("Feature Relevance (Mutual Information)", fontsize=13, fontweight="bold")
for bar, val in zip(bars, mi_series.values):
    ax.text(val + 0.0005, bar.get_y() + bar.get_height()/2,
            f"{val:.4f}", va="center", fontsize=8)
plt.tight_layout()
plt.show()


## 4. Export final feature set

In [ ]:
FINAL_FEATURES = [
    "credit_limit","age","debt_ratio","monthly_income",
    "num_open_credit_lines","num_real_estate_loans","num_dependents",
    "weighted_late_score","utilisation_ratio","income_band","dependents_income_ratio",
    "has_any_late","log_income","age_risk_flag","credit_lines_per_loan","high_debt_and_late",
    "target",
]

out_path = Path("../data/processed/features_final.csv")
df_eng[FINAL_FEATURES].to_csv(out_path, index=False)
print(f"Saved: {out_path}  |  {df_eng[FINAL_FEATURES].shape}")
print("\nNull counts:")
print(df_eng[FINAL_FEATURES].isnull().sum()[lambda s: s > 0].to_string() or "None")


**Next step → `03_modeling.ipynb`**